In [0]:
host = "dea.cgyi97rb4alr.us-east-1.rds.amazonaws.com"
port = "5432"
database = ""
user = ""
password = ""

jdbc_url = f"jdbc:postgresql://{host}:{port}/{database}"


In [0]:
from pyspark.sql import DataFrame

bronze_tables = [
    "all_payments",
    "calendly_scheduled_events",
    "close_crm_users_raw",
    "custom_activites_raw",
    "lead_activites_raw",
    "leads_raw",
    "mdl_users_raw",
    "student_sentiment",
    "lead_merges"
]

def load_postgres_table(
    source_table: str,
    target_table: str
):

    query = f"""
    (
        SELECT *
        FROM {source_table}
        WHERE DATE(INSERT_DATE) = CURRENT_DATE
    ) t
    """

    df = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", query)
        .option("user", user)
        .option("password", password)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    print(f"Loaded {source_table} -> {target_table}")

for t in bronze_tables:

    load_postgres_table(
        source_table=f"raw.{t}",
        target_table=f"customer_health.bronze.{t}"
    )

In [0]:
%sql
select max(insert_date) from customer_health.bronze.all_payments;